# Abalone Deep Learning Assignment
## 1. Regression: Rings 수치 예측
## 2. Classification: Rings를 3개 구간으로 나누어 분류

분류 구간 예시:
- class 0: Rings ≤ 8
- class 1: 9 ≤ Rings ≤ 10
- class 2: Rings ≥ 11

교수님이 별도 구간을 지정했다면 이 기준만 변경한다.


In [16]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, confusion_matrix, classification_report
)

tf.random.set_seed(42)
np.random.seed(42)


In [18]:
path = "/content/drive/MyDrive/Colab Notebooks/csv_file/abalone.csv"

cols = [
    "Sex", "Length", "Diameter", "Height",
    "Whole_weight", "Shucked_weight", "Viscera_weight",
    "Shell_weight", "Rings"
]

df = pd.read_csv(path)

# UCI 원본처럼 헤더가 없으면 다시 읽기
if "Rings" not in df.columns and "rings" not in [str(c).lower() for c in df.columns]:
    df = pd.read_csv(path, header=None, names=cols)

df.columns = [
    str(c).strip().replace(" ", "_").replace("-", "_")
    for c in df.columns
]

ring_col = next(c for c in df.columns if c.lower() == "rings")
sex_col = next((c for c in df.columns if c.lower() == "sex"), None)

print(df.head())
print(df.shape)
print(df.columns)


   id Sex  Length  Diameter  Height  Whole_weight  Shucked_weight  \
0   0   M   0.455     0.365   0.095        0.5140          0.2245   
1   1   M   0.350     0.265   0.090        0.2255          0.0995   
2   2   F   0.530     0.420   0.135        0.6770          0.2565   
3   3   M   0.440     0.365   0.125        0.5160          0.2155   
4   4   I   0.330     0.255   0.080        0.2050          0.0895   

   Viscera_weight  Shell_weight  Rings  
0          0.1010         0.150     15  
1          0.0485         0.070      7  
2          0.1415         0.210      9  
3          0.1140         0.155     10  
4          0.0395         0.055      7  
(4177, 10)
Index(['id', 'Sex', 'Length', 'Diameter', 'Height', 'Whole_weight',
       'Shucked_weight', 'Viscera_weight', 'Shell_weight', 'Rings'],
      dtype='object')


문자형 `Sex`는 One-hot encoding하고, `Rings`는 정답으로 분리한다.


In [19]:
X_base = df.drop([ring_col, "id"], axis=1).copy()

if sex_col is not None:
    X_base = pd.get_dummies(X_base, columns=[sex_col], dtype=int)

print(X_base.head())
print("입력 feature 수:", X_base.shape[1])


   Length  Diameter  Height  Whole_weight  Shucked_weight  Viscera_weight  \
0   0.455     0.365   0.095        0.5140          0.2245          0.1010   
1   0.350     0.265   0.090        0.2255          0.0995          0.0485   
2   0.530     0.420   0.135        0.6770          0.2565          0.1415   
3   0.440     0.365   0.125        0.5160          0.2155          0.1140   
4   0.330     0.255   0.080        0.2050          0.0895          0.0395   

   Shell_weight  Sex_F  Sex_I  Sex_M  
0         0.150      0      0      1  
1         0.070      0      0      1  
2         0.210      1      0      0  
3         0.155      0      0      1  
4         0.055      0      1      0  
입력 feature 수: 10


## 1. Regression: Rings 예측


In [20]:
X = X_base.values.astype(float)
y = df[ring_col].values.astype(float)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

scaler_reg = StandardScaler()
X_train = scaler_reg.fit_transform(X_train)
X_test = scaler_reg.transform(X_test)


In [21]:
model_reg = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(1)
])

model_reg.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

model_reg.summary()


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 64)             │           704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,817 (11.00 KB)

 Trainable params: 2,817 (11.00 KB)

 Non-trainable params: 0 (0.00 B)

In [22]:
history_reg = model_reg.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)


Epoch 1/50
84/84 ━━━━━━━━━━━━━━━━━━━━ 3s 23ms/step - loss: 38.4206 - mae: 4.8613 - val_loss: 11.3249 - val_mae: 2.4782
Epoch 2/50
84/84 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - loss: 9.6018 - mae: 2.1066 - val_loss: 6.2174 - val_mae: 1.8099
Epoch 3/50
84/84 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 6.9601 - mae: 1.7977 - val_loss: 5.2761 - val_mae: 1.6737
Epoch 4/50
84/84 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 6.1333 - mae: 1.7223 - val_loss: 4.9149 - val_mae: 1.6186
Epoch 5/50
84/84 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 5.7136 - mae: 1.6849 - val_loss: 4.7134 - val_mae: 1.5797
Epoch 6/50
84/84 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 5.4563 - mae: 1.6590 - val_loss: 4.5825 - val_mae: 1.5517
Epoch 7/50
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 5.2906 - mae: 1.6402 - val_loss: 4.4883 - val_mae: 1.5296
Epoch 8/50
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 5.1748 - mae: 1.6250 - val_loss: 4.4212 - val_mae: 1.5144
Epoch 9/50
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 5.1029 - 

In [23]:
y_pred = model_reg.predict(X_test).flatten()

print("MSE =", mean_squared_error(y_test, y_pred))
print("MAE =", mean_absolute_error(y_test, y_pred))
print("R2 =", r2_score(y_test, y_pred))


27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
MSE = 4.566709824726366
MAE = 1.5542269751215665
R2 = 0.5781412272364685


## 2. Classification
`Rings`를 3개 범주로 변환하여 다중분류 문제로 만든다.


In [24]:
ring_class = pd.cut(
    df[ring_col],
    bins=[-np.inf, 8, 10, np.inf],
    labels=[0, 1, 2]
).astype(int).values

X_cls = X_base.values.astype(float)
y_onehot = tf.keras.utils.to_categorical(ring_class, num_classes=3)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_cls, y_onehot,
    test_size=0.2,
    random_state=42,
    stratify=ring_class
)

scaler_cls = StandardScaler()
X_train_c = scaler_cls.fit_transform(X_train_c)
X_test_c = scaler_cls.transform(X_test_c)

print("class count:", np.bincount(ring_class))


class count: [1407 1323 1447]


In [25]:
model_cls = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_c.shape[1],)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(3, activation="softmax")
])

model_cls.compile(
    optimizer="sgd",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model_cls.summary()


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_9 (Dense)                 │ (None, 64)             │           704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,883 (11.26 KB)

 Trainable params: 2,883 (11.26 KB)

 Non-trainable params: 0 (0.00 B)

In [26]:
history_cls = model_cls.fit(
    X_train_c, y_train_c,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)


Epoch 1/50
84/84 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5026 - loss: 0.9825 - val_accuracy: 0.5964 - val_loss: 0.8580
Epoch 2/50
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5569 - loss: 0.8777 - val_accuracy: 0.6248 - val_loss: 0.8117
Epoch 3/50
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5760 - loss: 0.8436 - val_accuracy: 0.6323 - val_loss: 0.7918
Epoch 4/50
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5850 - loss: 0.8256 - val_accuracy: 0.6293 - val_loss: 0.7813
Epoch 5/50
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6007 - loss: 0.8149 - val_accuracy: 0.6338 - val_loss: 0.7749
Epoch 6/50
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6078 - loss: 0.8074 - val_accuracy: 0.6308 - val_loss: 0.7703
Epoch 7/50
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6145 - loss: 0.8015 - val_accuracy: 0.6338 - val_loss: 0.7667
Epoch 8/50
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6205 - loss: 0.7963 - val_accuracy: 0.6338 - val_loss:

In [27]:
y_prob_c = model_cls.predict(X_test_c)
y_pred_c = np.argmax(y_prob_c, axis=1)
y_true_c = np.argmax(y_test_c, axis=1)

print("Accuracy =", accuracy_score(y_true_c, y_pred_c))
print("\nConfusion Matrix")
print(confusion_matrix(y_true_c, y_pred_c))
print("\nClassification Report")
print(classification_report(y_true_c, y_pred_c))


27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Accuracy = 0.6232057416267942

Confusion Matrix
[[199  67  15]
 [ 52 143  70]
 [ 18  93 179]]

Classification Report
              precision    recall  f1-score   support

           0       0.74      0.71      0.72       281
           1       0.47      0.54      0.50       265
           2       0.68      0.62      0.65       290

    accuracy                           0.62       836
   macro avg       0.63      0.62      0.62       836
weighted avg       0.63      0.62      0.63       836

